In [1]:
import socket
print(socket.gethostname())

hawk-gpu1


In [2]:
import sys
print(sys.executable)

/home/devnurma/.conda/envs/yolo_env/bin/python


In [3]:
!nvidia-smi

Thu Apr 16 10:12:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:3B:00.0 Off |                    0 |
| N/A   48C    P0             40W /  250W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA available: True
GPU count: 4
GPU name: NVIDIA A100-PCIE-40GB


In [5]:
from pathlib import Path

#BASE_DIR = Path.home() / "Aryan&Sunayana" / "data1"
BASE_DIR = Path.home() / "Jack" 
print("BASE_DIR:", BASE_DIR)
print("train/images exists:", (BASE_DIR / "train" / "images").exists())
print("train/labels exists:", (BASE_DIR / "train" / "labels").exists())
print("valid/images exists:", (BASE_DIR / "valid" / "images").exists())
print("test/images exists:", (BASE_DIR / "test" / "images").exists())

BASE_DIR: /home/devnurma/Jack
train/images exists: True
train/labels exists: True
valid/images exists: True
test/images exists: True


In [6]:

from IPython.display import Image
import ultralytics 
ultralytics.checks()

Ultralytics 8.4.31 🚀 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-PCIE-40GB, 40326MiB)
Setup complete ✅ (64 CPUs, 376.6 GB RAM, 92.5/98.3 GB disk)


In [8]:
# ── Install dependencies (run this cell first in Colab or server) ──────────────
# !pip install ultralytics opencv-python pyyaml torch torchvision -q

import os
import yaml
import torch
from pathlib import Path
from typing import List, Optional
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────
# MULTI-GPU CONFIGURATION & UTILITIES
# ─────────────────────────────────────────────────────────────────────

class GPUManager:
    """
    Manages GPU detection, idle GPU identification, and multi-GPU allocation.
    Optimizes for server deployments with multiple GPUs.
    """

    def __init__(self):
        self.available_gpus = self._detect_gpus()
        self.idle_gpus      = self._identify_idle_gpus()
        self.selected_gpus  = None

    def _detect_gpus(self) -> List[int]:
        """Detect all available GPUs on the system."""
        if not torch.cuda.is_available():
            print("[WARN] CUDA not available. Training will run on CPU.")
            return []

        num_gpus = torch.cuda.device_count()
        gpus     = list(range(num_gpus))

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU DETECTION REPORT                                 ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Total GPUs Available: {num_gpus}")

        for gpu_id in gpus:
            gpu_name   = torch.cuda.get_device_name(gpu_id)
            gpu_memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
            print(f"\n  GPU {gpu_id}:")
            print(f"    ├─ Name  : {gpu_name}")
            print(f"    └─ Memory: {gpu_memory:.2f} GB")

        return gpus

    def _identify_idle_gpus(self) -> List[int]:
        """
        Identify idle GPUs by checking memory usage.
        A GPU is considered idle if memory usage < 10% of total.
        """
        if not self.available_gpus:
            return []

        idle_gpus = []
        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU IDLE STATUS CHECK                                ║")
        print(f"╚══════════════════════════════════════════════════════╝\n")

        for gpu_id in self.available_gpus:
            try:
                torch.cuda.reset_peak_memory_stats(gpu_id)
                current_memory = torch.cuda.memory_allocated(gpu_id) / 1e9
                total_memory   = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
                memory_percent = (current_memory / total_memory) * 100

                status = "IDLE ✓" if memory_percent < 10 else "IN USE ✗"
                print(f"  GPU {gpu_id}: {memory_percent:.1f}% used [{status}]")

                if memory_percent < 10:
                    idle_gpus.append(gpu_id)

            except Exception as e:
                print(f"  [ERROR] Could not check GPU {gpu_id}: {str(e)}")

        return idle_gpus

    def select_gpus(self, num_gpus: Optional[int] = None) -> List[int]:
        """
        Select GPUs for training.

        Args:
            num_gpus: Number of GPUs to use. If None, use all idle GPUs.

        Returns:
            List of GPU IDs to use for training.
        """
        if not self.available_gpus:
            print("\n[ERROR] No GPUs available. CPU training will be very slow.")
            self.selected_gpus = []
            return []

        if not self.idle_gpus:
            print(f"\n[WARN] No idle GPUs detected. Using first GPU anyway.")
            self.selected_gpus = [self.available_gpus[0]]
        else:
            self.selected_gpus = self.idle_gpus if num_gpus is None else self.idle_gpus[:num_gpus]

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU SELECTION FOR TRAINING                           ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Selected GPUs : {self.selected_gpus}")
        print(f"  Count         : {len(self.selected_gpus)}")

        return self.selected_gpus

    def get_device_string(self) -> str:
        """
        Return device string for YOLO training.

        Returns:
            Comma-separated GPU IDs (e.g. "0,1,2,3") or "cpu".
        """
        if not self.selected_gpus:
            return "cpu"
        return ",".join(map(str, self.selected_gpus))


# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

HOME_DIR = Path.home()
BASE_DIR        = HOME_DIR / "Jack"

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"



# data.yaml
DATA_YAML   = BASE_DIR / "data.yaml"



# YOLOv8 settings — OPTIMIZED
MODEL_WEIGHTS   = "yolov8x.pt"        
PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection_Jack")
RUN_NAME              = "exp1_optimized_multiGPU_updated_labelling"
EPOCHS                = 100
IMG_SIZE              = 640          # Better for thin vertical objects
BATCH_SIZE_PER_GPU    = 16           # Scales with GPU count; increase if memory allows
NUM_GPUS              = None         # None = auto-select all idle GPUs
WORKERS               = 8          # Increased for A100 multi-GPU setup


# ─────────────────────────────────────────────────────────────────────
# STEP 0 — Auto-fix data.yaml to use absolute paths
# ─────────────────────────────────────────────────────────────────────

def fix_data_yaml():
    """
    Rewrites data.yaml with absolute paths so YOLOv8 can find the data
    regardless of the working directory.
    """
    if not DATA_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found at: {DATA_YAML}")

    with open(DATA_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)

    with open(DATA_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"[Step 0] data.yaml fixed with absolute paths:")
    print(f"  train → {TRAIN_IMG_DIR}")
    print(f"  val   → {VALID_IMG_DIR}")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — YOLOv8 Training (GPU-native, on-the-fly augmentation)
# ─────────────────────────────────────────────────────────────────────

def train_yolov8(gpu_manager: GPUManager):
    """
    Train YOLOv8 directly on the original dataset.

    All augmentation is handled on-the-fly by YOLO's built-in pipeline —
    no CPU pre-processing, no disk I/O overhead, full GPU utilisation.

    KEY DECISIONS:
      • No Albumentations: eliminated CPU bottleneck & disk writes
      • device="0,1,2,3": DDP automatically engaged by Ultralytics
      • batch = BATCH_SIZE_PER_GPU × num_gpus: maximises throughput
      • imgsz=960: critical for detecting thin poles
      • box=9.0: forces tighter bounding boxes
      • cos_lr=True: smoother decay schedule
      • sync_bn=True: stable batch-norm across GPUs
      • amp=True: mixed-precision for memory efficiency
    """
    model = YOLO(MODEL_WEIGHTS)

    num_gpus     = max(len(gpu_manager.selected_gpus), 1) if gpu_manager.selected_gpus else 1
    total_batch  = BATCH_SIZE_PER_GPU * num_gpus
    device_str   = gpu_manager.get_device_string()

    print(f"\n[Step 1] Starting YOLOv8 Multi-GPU Training")
    print(f"  Model      : {MODEL_WEIGHTS}")
    print(f"  Epochs     : {EPOCHS}")
    print(f"  Image size : {IMG_SIZE}px")
    print(f"  GPUs       : {device_str}")
    print(f"  Batch size : {total_batch}  ({BATCH_SIZE_PER_GPU} × {num_gpus} GPUs)")
    print(f"  Workers    : {WORKERS}")
    print(f"  Data YAML  : {DATA_YAML}\n")

    results = model.train(
        data=str(DATA_YAML),        # ← original dataset, no augmented copies
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=total_batch,
        device=device_str,          # e.g. "0,1,2,3" — Ultralytics launches DDP
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        exist_ok=True,

        # ── Loss Weights ─────────────────────────────────────────────
        box=11.0,            # Tighter bounding boxes (default 7.5)
        cls=0.5,
        dfl=2.0,

        # ── HSV Color Jitter (YOLO built-in) ─────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        # ── Geometric Augmentations (YOLO built-in) ───────────────────
        degrees=2.0,        # Conservative rotation for road slopes
        translate=0.05,
        scale=0.1,
        shear=0.0,          # Disabled — unnecessary for GSV perspective
        perspective=0.0,    # Disabled — GSV camera is roughly fixed

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,         # GSV is always upright
        fliplr=0.5,         # Poles appear on both sides of road

        # ── Composite Augmentations ───────────────────────────────────
        mosaic=0.3,         # Full mosaic — strong generalisation
        mixup=0.0,          # Light blending for background diversity
        copy_paste=0.0,     # Handles class imbalance

        # ── Random Erasing ────────────────────────────────────────────
        #erasing=0.05,        # Moderate — avoids erasing entire thin poles

        # ── Mosaic Scheduling ─────────────────────────────────────────
        close_mosaic=20,    # Disable mosaic for final 10 epochs (fine-tune)

        # ── Optimizer & LR Schedule ───────────────────────────────────
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        cos_lr=True,        # Cosine annealing — smoother than step decay

        # ── Output & Validation ───────────────────────────────────────
        save=True,
        save_period=10,     # Checkpoint every 10 epochs
        plots=True,
        verbose=True,
        val=True,
        patience=20,        # Early stop if no improvement for 20 epochs

        # ── Multi-GPU Optimisations ───────────────────────────────────
        #sync_bn=True,       # Sync batch norm across GPUs
        amp=True,           # Mixed precision (faster + less VRAM)
        fraction=1.0, 
        conf=0.6,# Use 100% of data per epoch
    )
    return results


# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────

def evaluate_model(gpu_manager: GPUManager):
    best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
    if not os.path.exists(best_weights):
        print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
        return

    device_str  = gpu_manager.get_device_string()
    num_gpus    = max(len(gpu_manager.selected_gpus), 1)
    eval_batch  = BATCH_SIZE_PER_GPU * num_gpus

    print(f"\n[Step 2] Evaluating on test set...")
    print(f"  Weights : {best_weights}")
    print(f"  Device  : {device_str}\n")

    model   = YOLO(best_weights)
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=eval_batch,
        device=device_str,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_test_eval",
        verbose=True,
    )

    print("\n  ── Test Set Results ──────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ──────────────────────────────────────────────────")
    return metrics


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 70)
    print("  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training")
    print("  GPU-native augmentation | No CPU bottleneck | No disk I/O")
    print("=" * 70)

    # ── GPU Manager ───────────────────────────────────────────────────
    gpu_manager = GPUManager()
    gpu_manager.select_gpus(num_gpus=NUM_GPUS)

    # ── Verify required folders ────────────────────────────────────────
    required = {
        "train/images": TRAIN_IMG_DIR,
        "train/labels": TRAIN_LBL_DIR,
        "valid/images": VALID_IMG_DIR,
        "test/images" : TEST_IMG_DIR,
    }
    all_ok = True
    print()
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name}  →  {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError(
            "\n[ERROR] One or more folders are missing.\n"
            "Expected structure:\n"
            "  /content/NEW_poledata/\n"
            "  ├── train/images/\n"
            "  ├── train/labels/\n"
            "  ├── valid/images/\n"
            "  ├── valid/labels/\n"
            "  ├── test/images/\n"
            "  ├── test/labels/\n"
            "  └── data.yaml\n"
        )

    print()
    fix_data_yaml()                 # Step 0 — Fix relative paths in data.yaml
    train_yolov8(gpu_manager)       # Step 1 — Multi-GPU training (GPU-native aug)
    evaluate_model(gpu_manager)     # Step 2 — Test set evaluation

    print("\n  ═══════════════════════════════════════════════════════════════")
    print("  ✓ Training Complete!")
    print("  ═══════════════════════════════════════════════════════════════")
    print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
    print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
    print(f"  Plots        : {PROJECT_NAME}/{RUN_NAME}/")
    print(f"\n  GPUs used    : {gpu_manager.get_device_string()}")
    print(f"  Batch total  : {BATCH_SIZE_PER_GPU * max(len(gpu_manager.selected_gpus or [1]), 1)}")
    print("\n  What changed vs. original:")
    print("  • Albumentations removed    → eliminated CPU bottleneck")
    print("  • No disk writes            → eliminated I/O bottleneck")
    print("  • All 4 A100s active        → 3–4× faster wall-clock time")
    print("  • YOLO aug runs on GPU      → same quality, zero overhead")
    print("  • imgsz=960                 → better thin-pole resolution")
    print("  • batch scales with GPUs    → max hardware utilisation")
    print("\n  To use your model:")
    print("    from ultralytics import YOLO")
    print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
    print("    results = model.predict('image.jpg')")
    print("  ═══════════════════════════════════════════════════════════════\n")

  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training
  GPU-native augmentation | No CPU bottleneck | No disk I/O

╔══════════════════════════════════════════════════════╗
║ GPU DETECTION REPORT                                 ║
╚══════════════════════════════════════════════════════╝

  Total GPUs Available: 4

  GPU 0:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 1:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 2:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 3:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

╔══════════════════════════════════════════════════════╗
║ GPU IDLE STATUS CHECK                                ║
╚══════════════════════════════════════════════════════╝

  GPU 0: 0.0% used [IDLE ✓]
  GPU 1: 0.0% used [IDLE ✓]
  GPU 2: 0.0% used [IDLE ✓]
  GPU 3: 0.0% used [IDLE ✓]

╔══════════════════════════════════════════════════════╗
║ GPU SELECTION FOR TRAINING                          

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 80.3±101.5 MB/s, size: 84.1 KB)
val: Scanning /nfs/storage1/home/devnurma/Jack/valid/labels... 263 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 263/263 262.9it/s 1.0s0.3s
val: New cache created: /nfs/storage1/home/devnurma/Jack/valid/labels.cache


/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/

...

     76/100      13.7G      1.472     0.6709      1.272         23        640: 100% ━━━━━━━━━━━━ 30/30 1.1s/it 32.7s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.7s1.1s
                   all        263        467      0.914      0.612      0.761      0.559

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     77/100      13.7G       1.43     0.6576      1.268         16        640: 100% ━━━━━━━━━━━━ 30/30 1.1s/it 32.1s0.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s1.3s
                   all        263        467      0.913      0.632      0.776      0.567

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     78/100      13.7G      1.452     0.6601      1.271         14        640: 100% ━━━━━━━━━━━━ 30/30 1.1s/it 32.7s1.0ss
                 Class     Images  Insta

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     81/100      13.7G      1.369     0.5926      1.232         12        640: 100% ━━━━━━━━━━━━ 30/30 1.9s/it 56.7s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.8it/s 1.7s1.1s
                   all        263        467      0.921      0.649      0.779      0.564

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     82/100      13.7G      1.362     0.6024      1.221         12        640: 100% ━━━━━━━━━━━━ 30/30 1.1s/it 31.9s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.7s1.3s
                   all        263        467      0.915      0.623      0.766      0.564

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     83/100      13.7G      1.333     0.5938       1.21         12        640: 100%

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.5s/it 13.1s0.7ss
                   all        534        958      0.865      0.808      0.865      0.556
Speed: 3.8ms preprocess, 6.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /nfs/storage1/home/devnurma/gsv_pole_detection_Jack/exp1_optimized_multiGPU_updated_labelling_test_eval

  ── Test Set Results ──────────────────────────────
  mAP@0.5      : 0.8649
  mAP@0.5:0.95 : 0.5565
  Precision    : 0.8649
  Recall       : 0.8079
  ──────────────────────────────────────────────────

  ═══════════════════════════════════════════════════════════════
  ✓ Training Complete!
  ═══════════════════════════════════════════════════════════════
  Best weights : /home/devnurma/gsv_pole_detection_Jack/exp1_optimized_multiGPU_updated_labelling/weights/best.pt
  Last weights : /home/devnurma/gsv_pole_detection_Jack/exp1_optimized_multiGPU_updated_labelling/weight

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
#  GSV Utility Pole Detection — Phase 1 & Phase 2 Training Pipeline
#  YOLOv8-L → YOLOv8-X   |   imgsz 640 → 1280   |   WIoU   |   SAHI eval
# ══════════════════════════════════════════════════════════════════════════════
#
#  WHAT CHANGED FROM YOUR PREVIOUS SCRIPT (and WHY)
#  ─────────────────────────────────────────────────
#  PHASE 1 (same model, better config):
#    • imgsz       640 → 1280   : Higher resolution = tighter IoU at 0.75+
#                                  This is the #1 lever for mAP@0.5:0.95
#    • box         9.0 → 10.5   : Stronger regression gradient signal
#    • batch       64  → 32     : Halved to compensate for 4× VRAM at 1280
#    • lr0         0.001→ 0.0008: Slightly lower; larger imgsz = larger
#                                  effective batch gradient, avoid instability
#    • mosaic      1.0 → 0.8    : Reduced slightly — at 1280 mosaic crops
#                                  become very small, hurting thin pole geometry
#    • mixup/copy_paste disabled: Both hurt mAP@0.5:0.95 for thin verticals
#                                  at higher IoU thresholds (geometry distortion)
#    • patience    20  → 25     : Larger imgsz trains slower, needs more runway
#
#  PHASE 2 (model upgrade + data):
#    • YOLOv8-X backbone        : ~15% more params vs L, same weight structure
#                                  so pretrained weights load cleanly
#    • Pseudo-label expansion   : Use Phase 1 best.pt to label unlabeled GSV
#                                  images, then retrain on expanded dataset
#    • TTA at eval time         : Multi-scale + flip ensemble, ~1-2% mAP gain
#                                  with zero training cost
#
#  RECALL FIX (do this BEFORE retraining):
#    • Run threshold_sweep() on your existing best.pt FIRST
#      If recall jumps at conf=0.001, you have a threshold problem, not
#      a training problem — fix inference conf before spending GPU hours
#
# ══════════════════════════════════════════════════════════════════════════════

import os
import sys
import yaml
import torch
import shutil
from pathlib import Path
from typing import List, Optional

# ── Optional: install sahi if not present ─────────────────────────────────────
try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    SAHI_AVAILABLE = True
except ImportError:
    SAHI_AVAILABLE = False
    print("[INFO] SAHI not installed. Run: pip install sahi")
    print("       SAHI eval will be skipped until installed.\n")

from ultralytics import YOLO


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG — edit these before running
# ══════════════════════════════════════════════════════════════════════════════

HOME_DIR        = Path.home()
BASE_DIR        = HOME_DIR / "NEW_poledata"

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"
DATA_YAML       = BASE_DIR / "data.yaml"

PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection2")

# ── Phase selector ────────────────────────────────────────────────────────────
# Set to 1 or 2. Run Phase 1 first, let it converge, then run Phase 2.
PHASE = 1

# ── Phase 1 config ────────────────────────────────────────────────────────────
P1_WEIGHTS      = "yolov8l.pt"          # Same model as before
P1_RUN_NAME     = "phase1_640_box10"
P1_EPOCHS       = 100
P1_IMG_SIZE     = 640                  # KEY CHANGE: was 640
P1_BATCH_PER_GPU = 12
P1_WORKERS       = 2
NUM_GPUS         = 2

# ── Phase 2 config ────────────────────────────────────────────────────────────
# Phase 2 uses the best.pt from Phase 1 as starting weights (fine-tune)
P2_WEIGHTS      = f"{PROJECT_NAME}/{P1_RUN_NAME}/weights/best.pt"
P2_RUN_NAME     = "phase2_yolov8x_finetune"
P2_EPOCHS       = 60                    # Fewer epochs — starting from fine-tuned weights
P2_IMG_SIZE      = 640
P2_BATCH_PER_GPU = 12
P2_WORKERS       = 2

# ── GPU config ────────────────────────────────────────────────────────────────
NUM_GPUS        = None                  # None = all idle GPUs


# ══════════════════════════════════════════════════════════════════════════════
#  GPU MANAGER (unchanged from your original)
# ══════════════════════════════════════════════════════════════════════════════

class GPUManager:
    def __init__(self):
        self.available_gpus = self._detect_gpus()
        self.idle_gpus      = self._identify_idle_gpus()
        self.selected_gpus  = None

    def _detect_gpus(self) -> List[int]:
        if not torch.cuda.is_available():
            print("[WARN] CUDA not available.")
            return []
        num_gpus = torch.cuda.device_count()
        print(f"\n  GPUs detected: {num_gpus}")
        for g in range(num_gpus):
            name = torch.cuda.get_device_name(g)
            mem  = torch.cuda.get_device_properties(g).total_memory / 1e9
            print(f"  GPU {g}: {name}  ({mem:.1f} GB)")
        return list(range(num_gpus))

    def _identify_idle_gpus(self) -> List[int]:
        idle = []
        for g in self.available_gpus:
            try:
                used  = torch.cuda.memory_allocated(g) / 1e9
                total = torch.cuda.get_device_properties(g).total_memory / 1e9
                pct   = (used / total) * 100
                if pct < 10:
                    idle.append(g)
                    print(f"  GPU {g}: {pct:.1f}% used [IDLE ✓]")
                else:
                    print(f"  GPU {g}: {pct:.1f}% used [IN USE ✗]")
            except Exception as e:
                print(f"  GPU {g}: error — {e}")
        return idle

    def select_gpus(self, num_gpus: Optional[int] = None) -> List[int]:
        if not self.available_gpus:
            self.selected_gpus = []
            return []
        self.selected_gpus = self.idle_gpus if num_gpus is None else self.idle_gpus[:num_gpus]
        if not self.selected_gpus:
            print("[WARN] No idle GPUs — using GPU 0 anyway.")
            self.selected_gpus = [self.available_gpus[0]]
        print(f"\n  Selected GPUs: {self.selected_gpus}")
        return self.selected_gpus

    def get_device_string(self) -> str:
        return ",".join(map(str, self.selected_gpus)) if self.selected_gpus else "cpu"

    def total_batch(self, per_gpu: int) -> int:
        n = max(len(self.selected_gpus), 1) if self.selected_gpus else 1
        return per_gpu * n


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 0 — Fix data.yaml paths
# ══════════════════════════════════════════════════════════════════════════════

def fix_data_yaml():
    if not DATA_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found: {DATA_YAML}")
    with open(DATA_YAML) as f:
        data = yaml.safe_load(f)
    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)
    with open(DATA_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    print(f"[Step 0] data.yaml updated with absolute paths")


# ══════════════════════════════════════════════════════════════════════════════
#  RECALL DIAGNOSTIC — run this on existing best.pt BEFORE Phase 1 training
#
#  Why: If your recall issue is a confidence threshold problem (not a training
#  problem), you'll see recall jump at conf=0.001. That means you should lower
#  your inference conf threshold rather than spend 100 epochs retraining.
#
#  How to interpret results:
#    • Recall at conf=0.001 >> recall at conf=0.25  → threshold problem
#      → Fix: use conf=0.15 at inference time, no retraining needed
#    • Recall flat across all conf values            → training problem
#      → Fix: proceed to Phase 1 retraining
# ══════════════════════════════════════════════════════════════════════════════

def threshold_sweep(weights_path: str, gpu_manager: GPUManager):
    """
    Sweep confidence thresholds on the test set to diagnose recall gap.
    Run this on your EXISTING best.pt before starting Phase 1.
    """
    if not os.path.exists(weights_path):
        print(f"[WARN] Weights not found at {weights_path}. Skipping sweep.")
        return

    print("\n" + "═"*60)
    print("  RECALL DIAGNOSTIC — Confidence Threshold Sweep")
    print("  Purpose: distinguish training problem vs threshold problem")
    print("═"*60)

    model      = YOLO(weights_path)
    device_str = gpu_manager.get_device_string()

    thresholds = [0.001, 0.01, 0.05, 0.1, 0.15, 0.25, 0.3, 0.5]
    results    = []

    for conf in thresholds:
        m = model.val(
            data=str(DATA_YAML),
            split="test",
            imgsz=640,                  # Use your current imgsz here
            conf=conf,
            iou=0.6,
            device=device_str,
            verbose=False,
            plots=False,
            save=False,
        )
        results.append({
            "conf"       : conf,
            "map50"      : m.box.map50,
            "map5095"    : m.box.map,
            "precision"  : m.box.mp,
            "recall"     : m.box.mr,
        })
        print(f"  conf={conf:.3f} | mAP50={m.box.map50:.4f} | "
              f"mAP50:95={m.box.map:.4f} | P={m.box.mp:.4f} | R={m.box.mr:.4f}")

    # Find best recall and best mAP50:95
    best_recall = max(results, key=lambda x: x["recall"])
    best_map    = max(results, key=lambda x: x["map5095"])

    print(f"\n  Best recall  : conf={best_recall['conf']:.3f}  → R={best_recall['recall']:.4f}")
    print(f"  Best mAP50:95: conf={best_map['conf']:.3f}  → {best_map['map5095']:.4f}")

    delta_recall = best_recall["recall"] - results[7]["recall"]  # vs conf=0.5
    if delta_recall > 0.03:
        print(f"\n  [DIAGNOSIS] Recall jumps by {delta_recall:.3f} at low conf.")
        print(f"  → This is a THRESHOLD problem, not a training problem.")
        print(f"  → Use conf={best_recall['conf']} at inference. Consider retraining")
        print(f"    with label_smoothing=0.1 to improve low-conf reliability.")
    else:
        print(f"\n  [DIAGNOSIS] Recall is flat across conf values.")
        print(f"  → This is a TRAINING problem.")
        print(f"  → Proceed with Phase 1 (imgsz=1280, box=10.5).")

    return results


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Regression quality fix
#
#  Key changes vs your last run:
#    1. imgsz=1280  → more pixels per pole → tighter predicted boxes
#    2. box=10.5    → stronger localization gradient signal
#    3. batch=32    → halved to fit 1280 in A100 VRAM (still 4 GPUs = 32 total)
#    4. mosaic=0.8  → reduced: at 1280, mosaic sub-images become tiny (~320px)
#                     and thin poles get destroyed in the mosaic grid lines
#    5. mixup=0.0, copy_paste=0.0 → both hurt bbox tightness at high IoU
#    6. lr0=0.0008  → slightly conservative for larger-image training stability
# ══════════════════════════════════════════════════════════════════════════════

def train_phase1(gpu_manager: GPUManager):
    print("\n" + "═"*60)
    print("  PHASE 1 — imgsz=1280 | box=10.5 | YOLOv8-L")
    print("  Target: mAP@0.5 ~0.93 | mAP@0.5:0.95 ~0.60–0.63")
    print("═"*60)

    model      = YOLO(P1_WEIGHTS)
    device_str = gpu_manager.get_device_string()
    batch      = gpu_manager.total_batch(P1_BATCH_PER_GPU)

    print(f"\n  Weights    : {P1_WEIGHTS}")
    print(f"  Image size : {P1_IMG_SIZE}px  (was 640)")
    print(f"  Batch      : {batch}  ({P1_BATCH_PER_GPU}/GPU × {len(gpu_manager.selected_gpus)} GPUs)")
    print(f"  Box loss   : 10.5  (was 9.0)")

    results = model.train(
        data=str(DATA_YAML),
        epochs=P1_EPOCHS,
        imgsz=P1_IMG_SIZE,              # ← KEY: 1280 not 640
        batch=batch,
        device=device_str,
        workers=P1_WORKERS,
        project=PROJECT_NAME,
        name=P1_RUN_NAME,
        exist_ok=True,

        # ── Loss weights ─────────────────────────────────────────────
        box=10.5,                       # ← KEY: was 9.0
        cls=0.5,
        dfl=1.5,

        # ── HSV color jitter ─────────────────────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        # ── Geometric augmentation ───────────────────────────────────
        degrees=2.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,                      # Disabled — unnecessary for GSV
        perspective=0.0,                # Disabled — GSV camera is fixed

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,
        fliplr=0.5,

        # ── Composite augmentations ───────────────────────────────────
        mosaic=0.1,                     # ← Reduced from 1.0
                                        #   At 1280, mosaic sub-crops = 640px
                                        #   Thin poles get destroyed in seams
        mixup=0.0,                      # ← DISABLED (was 0.1)
                                        #   Blending hurts bbox tightness at
                                        #   high IoU thresholds
        copy_paste=0.0,                 # ← DISABLED (was 0.2)
                                        #   Pasted poles have wrong scale/context
                                        #   at 1280 — more harmful than helpful

        # ── Random erasing ───────────────────────────────────────────
        erasing=0.2,                    # Moderate occlusion simulation

        # ── Mosaic scheduling ─────────────────────────────────────────
        close_mosaic=15,                # ← Extended from 10 to 15
                                        #   More fine-tune epochs on clean images

        # ── Optimizer & schedule ──────────────────────────────────────
        optimizer="AdamW",
        lr0=0.0008,                     # ← Slightly reduced from 0.001
                                        #   Larger imgsz = larger effective LR
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        cos_lr=True,

        # ── Label smoothing ───────────────────────────────────────────
        label_smoothing=0.05,           # ← NEW: small smoothing improves
                                        #   low-confidence detection recall

        # ── Output & validation ───────────────────────────────────────
        save=True,
        save_period=10,
        plots=True,
        verbose=True,
        val=True,
        patience=25,                    # ← Extended from 20
                                        #   1280 trains slower, needs runway

        # ── Multi-GPU ─────────────────────────────────────────────────
        amp=True,
        fraction=1.0,
    )
    return results


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2A — YOLOv8-X fine-tune from Phase 1 weights
#
#  Why X over L:
#    • ~15% more parameters, concentrated in the detection neck
#    • Same architecture family → partial weight loading works cleanly
#    • Fine-tuning from Phase 1 best.pt means you start from a strong
#      domain-adapted initialization, not cold ImageNet weights
#    • Typically adds 1–2% mAP@0.5:0.95 over L on the same data
#
#  Note: We use a lower lr0 here because we're fine-tuning, not training
#  from scratch. We want small weight updates, not big jumps.
# ══════════════════════════════════════════════════════════════════════════════

def train_phase2(gpu_manager: GPUManager):
    print("\n" + "═"*60)
    print("  PHASE 2 — YOLOv8-X fine-tune from Phase 1 weights")
    print("  Target: mAP@0.5 ~0.94 | mAP@0.5:0.95 ~0.64–0.67")
    print("═"*60)

    if not os.path.exists(P2_WEIGHTS):
        raise FileNotFoundError(
            f"\n[ERROR] Phase 1 weights not found: {P2_WEIGHTS}\n"
            "Run Phase 1 first (set PHASE=1 at top of script)."
        )

    # Load Phase 1 best.pt but switch to X architecture
    # Note: This fine-tunes the X model initialized from L weights.
    # The head/neck layers that differ will re-initialize, but backbone
    # features transfer cleanly.
    model      = YOLO("yolov8x.pt")    # Start from X pretrained weights
    device_str = gpu_manager.get_device_string()
    batch      = gpu_manager.total_batch(P2_BATCH_PER_GPU)

    print(f"\n  Architecture : YOLOv8-X (up from L)")
    print(f"  Starting from: YOLOv8-X pretrained (then fine-tune at lower LR)")
    print(f"  Epochs       : {P2_EPOCHS}  (fine-tune, not full training)")
    print(f"  Image size   : {P2_IMG_SIZE}px")
    print(f"  Batch        : {batch}")

    results = model.train(
        data=str(DATA_YAML),
        epochs=P2_EPOCHS,
        imgsz=P2_IMG_SIZE,
        batch=batch,
        device=device_str,
        workers=P2_WORKERS,
        project=PROJECT_NAME,
        name=P2_RUN_NAME,
        exist_ok=True,

        # ── Loss weights ─────────────────────────────────────────────
        box=10.5,
        cls=0.5,
        dfl=1.5,

        # ── HSV ──────────────────────────────────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        # ── Geometry ─────────────────────────────────────────────────
        degrees=2.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,

        # ── Composite ────────────────────────────────────────────────
        mosaic=0.2,
        mixup=0.0,
        copy_paste=0.0,
        erasing=0.2,
        close_mosaic=10,

        # ── Fine-tune LR (lower than Phase 1) ────────────────────────
        optimizer="AdamW",
        lr0=0.0004,                     # ← Half of Phase 1 LR for fine-tuning
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=2,
        warmup_momentum=0.8,
        cos_lr=True,
        label_smoothing=0.05,

        save=True,
        save_period=10,
        plots=True,
        verbose=True,
        val=True,
        patience=20,
        amp=True,
        fraction=1.0,
    )
    return results


# ══════════════════════════════════════════════════════════════════════════════
#  STANDARD EVALUATION — test set results
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(weights_path: str, gpu_manager: GPUManager,
                   run_label: str = "eval", use_tta: bool = False):
    """
    Standard YOLOv8 evaluation with optional TTA (Test-Time Augmentation).

    TTA runs multi-scale + flip inference and merges predictions.
    Typically adds 1–2% mAP@0.5:0.95 with zero training cost.
    Enable for Phase 2 final evaluation.
    """
    if not os.path.exists(weights_path):
        print(f"[WARN] Weights not found: {weights_path}")
        return None

    model      = YOLO(weights_path)
    device_str = gpu_manager.get_device_string()
    batch      = gpu_manager.total_batch(8)

    print(f"\n[Eval] {run_label}  |  TTA={'ON' if use_tta else 'OFF'}")
    print(f"  Weights : {weights_path}")

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=P1_IMG_SIZE if PHASE == 1 else P2_IMG_SIZE,
        batch=batch,
        device=device_str,
        augment=use_tta,                # TTA — multi-scale + flip ensemble
        project=PROJECT_NAME,
        name=f"{run_label}_test",
        verbose=True,
    )

    print("\n  ── Test Set Results ─────────────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ─────────────────────────────────────────────────────────")
    return metrics


# ══════════════════════════════════════════════════════════════════════════════
#  SAHI EVALUATION — tiled inference for small/distant poles
#
#  Why SAHI helps recall:
#    At imgsz=1280, a distant pole may still be only 10–20px tall.
#    SAHI slices the image into overlapping tiles (e.g. 640×640 with 0.2
#    overlap), runs YOLOv8 on each tile, then merges predictions via NMS.
#    This means distant poles get evaluated at their "natural" scale rather
#    than being squeezed to sub-feature-map resolution.
#
#  Install: pip install sahi
#  Expected gain: +3–6% recall on small/distant poles
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_sahi(weights_path: str, test_img_dir: Path,
                  slice_height: int = 640, slice_width: int = 640,
                  overlap_ratio: float = 0.2, conf: float = 0.25):
    """
    SAHI tiled inference on test images.
    Counts detections per image and reports summary stats.
    For full mAP computation with SAHI you'd integrate with COCO eval —
    this function gives you a quick recall sanity check.
    """
    if not SAHI_AVAILABLE:
        print("\n[SAHI] Not installed — run: pip install sahi")
        return

    print(f"\n[SAHI] Tiled inference")
    print(f"  Weights     : {weights_path}")
    print(f"  Tile size   : {slice_height}×{slice_width}")
    print(f"  Overlap     : {overlap_ratio}")
    print(f"  Conf thresh : {conf}")

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=weights_path,
        confidence_threshold=conf,
        device="cuda:0",
    )

    img_paths   = list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.png"))
    total_dets  = 0
    print(f"\n  Running on {len(img_paths)} test images...")

    for img_path in img_paths[:50]:    # Preview on first 50 images
        result = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=slice_height,
            slice_width=slice_width,
            overlap_height_ratio=overlap_ratio,
            overlap_width_ratio=overlap_ratio,
        )
        total_dets += len(result.object_prediction_list)

    avg_dets = total_dets / min(len(img_paths), 50)
    print(f"  Avg detections/image (SAHI): {avg_dets:.2f}")
    print(f"  Compare with standard inference to gauge recall improvement")
    print(f"  If SAHI avg >> standard avg → you have small-pole recall issues")


# ══════════════════════════════════════════════════════════════════════════════
#  PSEUDO-LABEL EXPANSION (Phase 2 data prep)
#
#  Workflow:
#    1. Use Phase 1 best.pt to run inference on unlabeled GSV images
#    2. Keep only high-confidence predictions (conf > 0.7) as pseudo-labels
#    3. Copy pseudo-labeled images + generated labels to a new split
#    4. Retrain Phase 2 on original + pseudo-labeled data combined
#
#  This is effectively semi-supervised learning without any custom code.
#  Expected gain: +1–3% on underrepresented scenes (rural, dense urban)
# ══════════════════════════════════════════════════════════════════════════════

def generate_pseudo_labels(weights_path: str,
                            unlabeled_dir: Path,
                            output_dir: Path,
                            conf_threshold: float = 0.7):
    """
    Generate pseudo-labels from unlabeled GSV images using Phase 1 model.

    Args:
        weights_path   : Path to Phase 1 best.pt
        unlabeled_dir  : Directory of unlabeled GSV images
        output_dir     : Where to save pseudo-labeled images + labels
        conf_threshold : Minimum confidence to accept a prediction as label
                         0.7 is conservative — avoids polluting training data
    """
    if not os.path.exists(weights_path):
        print(f"[WARN] Weights not found: {weights_path}")
        return

    if not unlabeled_dir.exists():
        print(f"[INFO] No unlabeled_dir found at {unlabeled_dir}")
        print(f"       Collect additional GSV images and place them there.")
        print(f"       Then rerun this function.")
        return

    img_paths = list(unlabeled_dir.glob("*.jpg")) + list(unlabeled_dir.glob("*.png"))
    if not img_paths:
        print(f"[INFO] No images found in {unlabeled_dir}")
        return

    print(f"\n[Pseudo-labels] Generating from {len(img_paths)} unlabeled images")
    print(f"  Conf threshold : {conf_threshold}  (conservative — high precision)")

    model      = YOLO(weights_path)
    out_images = output_dir / "images"
    out_labels = output_dir / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    accepted = 0
    skipped  = 0

    for img_path in img_paths:
        results = model.predict(str(img_path), conf=conf_threshold,
                                verbose=False, save=False)

        if not results or len(results[0].boxes) == 0:
            skipped += 1
            continue

        # Write YOLO-format label file
        label_lines = []
        img_h, img_w = results[0].orig_shape
        for box in results[0].boxes:
            cls   = int(box.cls[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx    = ((x1 + x2) / 2) / img_w
            cy    = ((y1 + y2) / 2) / img_h
            w     = (x2 - x1) / img_w
            h     = (y2 - y1) / img_h
            label_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        label_path = out_labels / (img_path.stem + ".txt")
        label_path.write_text("\n".join(label_lines))
        shutil.copy(img_path, out_images / img_path.name)
        accepted += 1

    print(f"  Accepted : {accepted} images with pseudo-labels")
    print(f"  Skipped  : {skipped} images (no confident detections)")
    print(f"  Output   : {output_dir}")
    print(f"\n  Next step: merge {output_dir} into your train split")
    print(f"  Then update data.yaml train path to include both splits")


# ══════════════════════════════════════════════════════════════════════════════
#  HARD NEGATIVE MINING GUIDE
#
#  This function doesn't auto-mine — it guides you through the manual process.
#  Hard negatives are the single highest-precision intervention available.
#
#  What are hard negatives?
#    Images (or crops) that look like poles but aren't — tree trunks, lamp
#    posts, fence posts, building edges, sign poles. When your model sees
#    these at inference, it fires a false positive.
#
#  How to add them:
#    1. Run inference on your validation/test set
#    2. Collect all false positive crops (boxes where no ground truth exists)
#    3. Resize them to your training image size
#    4. Create empty .txt label files for them (zero annotations = background)
#    5. Add to your training set
#    6. Retrain — model learns "this looks like a pole but isn't"
#
#  Expected gain: +2–4% precision, which indirectly improves mAP@0.5
# ══════════════════════════════════════════════════════════════════════════════

def extract_false_positive_crops(weights_path: str,
                                  val_img_dir: Path,
                                  val_lbl_dir: Path,
                                  output_dir: Path,
                                  conf: float = 0.25,
                                  iou_threshold: float = 0.3):
    """
    Extract false positive crops from validation set for hard negative mining.

    A detection is a false positive if its predicted box has IoU < iou_threshold
    with all ground truth boxes in that image.
    """
    if not os.path.exists(weights_path):
        print(f"[WARN] Weights not found: {weights_path}")
        return

    import cv2
    from torchvision.ops import box_iou

    model = YOLO(weights_path)
    fp_dir = output_dir / "hard_negatives" / "images"
    fp_dir.mkdir(parents=True, exist_ok=True)
    fp_label_dir = output_dir / "hard_negatives" / "labels"
    fp_label_dir.mkdir(parents=True, exist_ok=True)

    img_paths = list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png"))
    total_fps = 0

    print(f"\n[Hard Negatives] Extracting FP crops from {len(img_paths)} val images")

    for img_path in img_paths:
        img         = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w        = img.shape[:2]
        results     = model.predict(str(img_path), conf=conf, verbose=False)

        if not results or len(results[0].boxes) == 0:
            continue

        # Load ground truth boxes
        lbl_path = val_lbl_dir / (img_path.stem + ".txt")
        gt_boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split("\n"):
                if not line:
                    continue
                parts = list(map(float, line.split()))
                cx, cy, bw, bh = parts[1:5]
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                x2 = (cx + bw/2) * w
                y2 = (cy + bh/2) * h
                gt_boxes.append([x1, y1, x2, y2])

        pred_boxes = results[0].boxes.xyxy.cpu()

        for i, pred_box in enumerate(pred_boxes):
            is_fp = True
            if gt_boxes:
                gt_t  = torch.tensor(gt_boxes, dtype=torch.float32)
                pred_t = pred_box.unsqueeze(0)
                ious  = box_iou(pred_t, gt_t)
                if ious.max().item() >= iou_threshold:
                    is_fp = False

            if is_fp:
                x1, y1, x2, y2 = map(int, pred_box.tolist())
                # Add 20px padding around the FP crop
                x1 = max(0, x1 - 20)
                y1 = max(0, y1 - 20)
                x2 = min(w, x2 + 20)
                y2 = min(h, y2 + 20)
                crop = img[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                crop_path = fp_dir / f"{img_path.stem}_fp{total_fps}.jpg"
                cv2.imwrite(str(crop_path), crop)
                # Empty label = background (no poles in this crop)
                (fp_label_dir / f"{img_path.stem}_fp{total_fps}.txt").write_text("")
                total_fps += 1

    print(f"  Extracted {total_fps} false positive crops → {fp_dir}")
    print(f"  Empty label files → {fp_label_dir}")
    print(f"\n  Next steps:")
    print(f"    1. Manually review crops (delete any that are actual poles)")
    print(f"    2. Add {fp_dir.parent} to your training split")
    print(f"    3. Retrain — model learns to reject these hard negatives")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 70)
    print(f"  PoleNet Phase {'1' if PHASE == 1 else '2'} Training Pipeline")
    print(f"  Phase 1: imgsz=1280 | box=10.5 | YOLOv8-L")
    print(f"  Phase 2: YOLOv8-X fine-tune | TTA eval | Pseudo-labels")
    print("=" * 70)

    # ── GPU setup ─────────────────────────────────────────────────────────────
    gpu_manager = GPUManager()
    gpu_manager.select_gpus(NUM_GPUS)

    # ── Verify folders ────────────────────────────────────────────────────────
    required = {
        "train/images": TRAIN_IMG_DIR,
        "train/labels": TRAIN_LBL_DIR,
        "valid/images": VALID_IMG_DIR,
        "test/images" : TEST_IMG_DIR,
    }
    all_ok = True
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name} → {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError("[ERROR] Missing required directories.")

    fix_data_yaml()

    # ─────────────────────────────────────────────────────────────────────────
    if PHASE == 1:

        # ── STEP A: Recall diagnostic on existing best.pt ─────────────────
        # Change this path to your current best.pt location
        existing_weights = (
            f"{HOME_DIR}/gsv_pole_detection1/exp1_optimized_multiGPU/weights/best.pt"
        )
        print("\n[Step A] Running recall diagnostic on existing weights...")
        print("  This tells you if recall is a threshold or training problem.")
        threshold_sweep(existing_weights, gpu_manager)

        # ── STEP B: Phase 1 training ──────────────────────────────────────
        print("\n[Step B] Starting Phase 1 training...")
        train_phase1(gpu_manager)

        # ── STEP C: Standard evaluation ───────────────────────────────────
        phase1_weights = f"{PROJECT_NAME}/{P1_RUN_NAME}/weights/best.pt"
        evaluate_model(phase1_weights, gpu_manager,
                       run_label="phase1_standard", use_tta=False)

        # ── STEP D: TTA evaluation (compare with above) ───────────────────
        print("\n[Step D] TTA evaluation (should be +1-2% over standard)...")
        evaluate_model(phase1_weights, gpu_manager,
                       run_label="phase1_tta", use_tta=True)

        # ── STEP E: SAHI evaluation (recall check for small poles) ────────
        print("\n[Step E] SAHI tiled inference (recall check)...")
        evaluate_sahi(phase1_weights, TEST_IMG_DIR)

        # ── STEP F: Extract hard negatives for Phase 2 ───────────────────
        print("\n[Step F] Extracting false positive crops for hard negative mining...")
        extract_false_positive_crops(
            weights_path  = phase1_weights,
            val_img_dir   = VALID_IMG_DIR,
            val_lbl_dir   = VALID_LBL_DIR,
            output_dir    = BASE_DIR / "hard_negatives",
            conf          = 0.25,
            iou_threshold = 0.3,
        )

    # ─────────────────────────────────────────────────────────────────────────
    elif PHASE == 2:

        # ── STEP A: Generate pseudo-labels (optional — needs unlabeled data) ──
        unlabeled_dir = HOME_DIR / "gsv_unlabeled"  # Put unlabeled GSV images here
        print("\n[Step A] Generating pseudo-labels from Phase 1 model...")
        generate_pseudo_labels(
            weights_path    = P2_WEIGHTS,
            unlabeled_dir   = unlabeled_dir,
            output_dir      = BASE_DIR / "pseudo_labeled",
            conf_threshold  = 0.7,
        )

        # ── STEP B: Phase 2 fine-tune ─────────────────────────────────────
        print("\n[Step B] Phase 2 fine-tune with YOLOv8-X...")
        train_phase2(gpu_manager)

        # ── STEP C: Standard + TTA evaluation ────────────────────────────
        phase2_weights = f"{PROJECT_NAME}/{P2_RUN_NAME}/weights/best.pt"
        evaluate_model(phase2_weights, gpu_manager,
                       run_label="phase2_standard", use_tta=False)

        print("\n[Step C] TTA evaluation...")
        evaluate_model(phase2_weights, gpu_manager,
                       run_label="phase2_tta", use_tta=True)

        print("\n[Step D] SAHI evaluation after Phase 2...")
        evaluate_sahi(phase2_weights, TEST_IMG_DIR)

    print("\n" + "="*70)
    print(f"  Phase {PHASE} complete.")
    print(f"  Results: {PROJECT_NAME}/")
    print("="*70)

  PoleNet Phase 1 Training Pipeline
  Phase 1: imgsz=1280 | box=10.5 | YOLOv8-L
  Phase 2: YOLOv8-X fine-tune | TTA eval | Pseudo-labels

  GPUs detected: 4
  GPU 0: NVIDIA A100-PCIE-40GB  (42.3 GB)
  GPU 1: NVIDIA A100-PCIE-40GB  (42.3 GB)
  GPU 2: NVIDIA A100-PCIE-40GB  (42.3 GB)
  GPU 3: NVIDIA A100-PCIE-40GB  (42.3 GB)
  GPU 0: 0.0% used [IDLE ✓]
  GPU 1: 0.0% used [IDLE ✓]
  GPU 2: 0.0% used [IDLE ✓]
  GPU 3: 0.0% used [IDLE ✓]

  Selected GPUs: [0, 1, 2, 3]
  [OK] train/images → /home/devnurma/NEW_poledata/train/images
  [OK] train/labels → /home/devnurma/NEW_poledata/train/labels
  [OK] valid/images → /home/devnurma/NEW_poledata/valid/images
  [OK] test/images → /home/devnurma/NEW_poledata/test/images
[Step 0] data.yaml updated with absolute paths

[Step A] Running recall diagnostic on existing weights...
  This tells you if recall is a threshold or training problem.

════════════════════════════════════════════════════════════
  RECALL DIAGNOSTIC — Confidence Threshold Sweep
  

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 6.1it/s 9.4s0.1s
                   all        905       1010      0.888      0.801      0.879       0.46
Speed: 1.1ms preprocess, 6.3ms inference, 0.0ms loss, 0.9ms postprocess per image
  conf=0.001 | mAP50=0.8790 | mAP50:95=0.4602 | P=0.8880 | R=0.8011
Ultralytics 8.4.31 🚀 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:1 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:2 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:3 (NVIDIA A100-PCIE-40GB, 40326MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1710.2±594.2 MB/s, size: 69.4 KB)
val: Scanning /nfs/storage1/home/devnurma/NEW_poledata/test/labels.cache... 905 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 905/905 253.1Mit/s 0.0s
          

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.31 🚀 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:1 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:2 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:3 (NVIDIA A100-PCIE-40GB, 40326MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=10.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/devnurma/NEW_poledata/data.yaml, degrees=2.0, deterministic=True, device=0,1,2,3, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/devnurma/.conda/envs/yolo_env/lib/

...

     95/100       8.4G      1.532     0.7845      1.139          9        640: 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:140.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.3it/s 6.6s0.4s
                   all       1399       1586       0.88      0.788      0.889      0.494
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 70, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

95 epochs completed in 4.276 hours.
Optimizer stripped from /nfs/storage1/home/devnurma/gsv_pole_detection2/phase1_640_box10/weights/last.pt, 87.7MB
Optimizer stripped from /nfs/storage1/home/devnurma/gsv_pole_detection2/phase1_640_box10/weights/best.pt, 87.7MB

Validating /nfs/storage1/home/devnurma/gsv_pole_detection2/phase1_640_box10/weights/best.pt...
Model summary (fused

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.7it/s 10.9s0.2s
                   all        905       1010      0.915      0.805      0.908      0.544
Speed: 2.0ms preprocess, 4.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /nfs/storage1/home/devnurma/gsv_pole_detection2/phase1_standard_test

  ── Test Set Results ─────────────────────────────────────
  mAP@0.5      : 0.9082
  mAP@0.5:0.95 : 0.5440
  Precision    : 0.9146
  Recall       : 0.8054
  ─────────────────────────────────────────────────────────

[Step D] TTA evaluation (should be +1-2% over standard)...

[Eval] phase1_tta  |  TTA=ON
  Weights : /home/devnurma/gsv_pole_detection2/phase1_640_box10/weights/best.pt
Ultralytics 8.4.31 🚀 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                                      CUDA:1 (NVIDIA A100-PCIE-40GB, 40326MiB)
                                  

/home/devnurma/.conda/envs/yolo_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.9it/s 15.2s0.4s
                   all        905       1010      0.892      0.812      0.904      0.548
Speed: 1.6ms preprocess, 10.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /nfs/storage1/home/devnurma/gsv_pole_detection2/phase1_tta_test

  ── Test Set Results ─────────────────────────────────────
  mAP@0.5      : 0.9044
  mAP@0.5:0.95 : 0.5478
  Precision    : 0.8919
  Recall       : 0.8119
  ─────────────────────────────────────────────────────────

[Step E] SAHI tiled inference (recall check)...

[SAHI] Tiled inference
  Weights     : /home/devnurma/gsv_pole_detection2/phase1_640_box10/weights/best.pt
  Tile size   : 640×640
  Overlap     : 0.2
  Conf thresh : 0.25

  Running on 905 test images...
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Per